# 02 — Sessionization & pairing pilot (CPU)
Jalankan berurutan, runtime CPU. **Tidak perlu mengulang notebook 01.**
Sumber: laporan P0 yang sudah berhasil, keenam NPZ, empat PCAP USTC kecil,
dan arsip Malicious_TLS (hanya daftar isi, tidak diekstrak/dijalankan).
Unduhan dibatasi 400 MiB NPZ + 32 MiB PCAP + 20 MiB RAR.
Membentuk citra dan [payload length, arah, IAT] dari 8 paket sesi yang SAMA,
lalu mencari kecocokan hash terhadap NPZ lama. Aturan sesi kandidat: biflow,
idle >60 detik, SYN baru (sequence berbeda), dan sesudah RST. Ini bukan klaim
bahwa pembagian sesi asli telah direproduksi. Hanya first-8, bukan sliding windows.
Output masih kandidat, **bukan dataset siap training**, tanpa split atau label tebakan.
Jika tidak cocok, cek preprocessing/sessionization/provenance; jangan paksakan mapping.


In [ ]:
%pip -q install scapy==2.5.0 rarfile==4.2
import sys, json, hashlib, io, uuid
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
auth.authenticate_user()
credentials, _ = google.auth.default()
drive = build('drive', 'v3', credentials=credentials, cache_discovery=False)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
WORK = Path('/content/temporal_pilot_p1') / RUN_ID
WORK.mkdir(parents=True, exist_ok=False)
print('CPU pairing pilot:', RUN_ID)


In [ ]:
SOURCES = {'pilot/__init__.py': '"""Isolated temporal-fusion pilot; does not modify the replication pipeline."""\n', 'pilot/pairing.py': '"""P1 candidate construction, not proof of the original sessionization protocol."""\nimport hashlib\nimport json\nfrom collections import defaultdict\nfrom decimal import Decimal\nfrom pathlib import Path\n\nimport numpy as np\n\nPOLICY = \'biflow-idle60s-syn-seq-rst-v1-first8\'\n# Evidence: data/splits.py comment. Verify against labels of actual hash matches.\nUSTC_LABELS = dict(zip((\'Gmail FTP Nsis-ay Facetime Weibo Cridex Zeus SMB BitTorrent \'\n                        \'WorldOfWarcraft Shifu Outlook Virut Geodo MySQL Htbot Tinba Skype Miuref Neris\').split(), range(20)))\n\n\ndef sha_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024**2), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef make_index(paths):\n    """No pooling of label namespaces, no guessed row correspondence."""\n    index = defaultdict(list)\n    sources = []\n    for path in paths:\n        path = Path(path)\n        namespace = \'USTC\' if \'USTC_1c_\' in path.name else \'combined\' if \'combined_\' in path.name else \'mal\'\n        with np.load(path, allow_pickle=False) as data:\n            x, y = data[\'data\'], data[\'target\']\n            if x.dtype != np.uint8 or x.shape[1:] != (32, 32) or y.shape != (len(x),):\n                raise ValueError(f\'Unsupported NPZ schema: {path.name}\')\n            if not np.issubdtype(y.dtype, np.integer):\n                raise ValueError(\'Non-integer labels\')\n            for row, (image, label) in enumerate(zip(x, y)):\n                index[hashlib.sha256(image.tobytes()).hexdigest()].append(\n                    {\'dataset\': namespace, \'file\': path.name, \'row\': row, \'label\': int(label)})\n            sources.append({\'file\': path.name, \'sha256\': sha_file(path), \'rows\': len(x), \'dataset\': namespace})\n    return index, sources\n\n\ndef extract_candidates(path, capture_class, packet_limit=200000, session_limit=10000):\n    """Read in capture order; preserve exact decimal times before IAT subtraction.\n\n    Biflow key: IP version, TCP/UDP, unordered endpoint pair. Split on idle>60s,\n    a new initiating SYN sequence (not retransmissions), or first packet after RST.\n    FIN alone does not split: its ACK may still belong to that session. This is an\n    explicit candidate policy, not a claim to reproduce an unknown original tool.\n    """\n    from scapy.all import IP, IPv6, TCP, Padding, PcapReader\n    from data.Preprocessing.utils import raw_packet_to_string\n    path = Path(path)\n    capture_hash = sha_file(path)\n    states, sessions, errors = {}, [], []\n    scanned = excluded = 0\n    complete = True\n    with PcapReader(str(path)) as packets:\n        for packet_id, packet in enumerate(packets):\n            if scanned >= packet_limit:\n                complete = False\n                break\n            scanned += 1\n            try:\n                header, payload = raw_packet_to_string(packet)\n            except ValueError:\n                excluded += 1\n                continue\n            ip = packet[IP] if IP in packet else packet[IPv6]\n            transport = ip[TCP] if TCP in ip else ip.payload\n            # IPv6 extension headers: select actual UDP layer explicitly.\n            if TCP not in ip:\n                from scapy.all import UDP\n                transport = ip[UDP]\n            src, dst = (ip.src, int(transport.sport)), (ip.dst, int(transport.dport))\n            key = (ip.version, \'TCP\' if TCP in ip else \'UDP\', tuple(sorted((src, dst))))\n            now = Decimal(str(packet.time))\n            if not now.is_finite():\n                raise ValueError(f\'Nonfinite capture timestamp at packet {packet_id}\')\n            syn = (src, int(transport.seq)) if TCP in ip and transport.flags.S and not transport.flags.A else None\n            state = states.get(key)\n            if state is not None and now < state[\'last\']:\n                # Fail closed for the capture: a later invalid packet must not\n                # leave an earlier prefix looking like a verified session.\n                raise ValueError(f\'Nonmonotonic per-biflow timestamp at packet {packet_id}\')\n            new = state is None or now - state[\'last\'] > 60 or state[\'reset\'] or (\n                syn is not None and syn != state[\'syn\'])\n            if new:\n                if len(sessions) >= session_limit:\n                    complete = False\n                    break\n                identity = json.dumps([capture_hash, POLICY, key, packet_id], separators=(\',\', \':\'))\n                state = {\'id\': hashlib.sha256(identity.encode()).hexdigest(), \'first\': src,\n                         \'last\': now, \'syn\': syn, \'reset\': False, \'blocks\': [], \'seq\': [],\n                         \'packet_ids\': [], \'times\': [], \'total\': 0}\n                sessions.append(state)\n                states[key] = state\n            if len(state[\'blocks\']) < 8:\n                copy = transport.copy()\n                if Padding in copy:\n                    copy[Padding].underlayer.remove_payload()\n                delta = Decimal(0) if not state[\'times\'] else now - Decimal(state[\'times\'][-1])\n                state[\'seq\'].append([len(bytes(copy.payload)), int(src != state[\'first\']), float(delta)])\n                state[\'blocks\'].append(header + payload)\n                state[\'packet_ids\'].append(packet_id)\n                state[\'times\'].append(str(now))\n            state[\'last\'] = now\n            state[\'total\'] += 1\n            state[\'reset\'] = bool(TCP in ip and transport.flags.R)\n    records, images, sequences, lengths = [], [], [], []\n    for state in sessions:\n        n = len(state[\'blocks\'])\n        image = np.frombuffer(bytes.fromhex(\'\'.join(state[\'blocks\'])).ljust(1024, b\'\\x00\'), dtype=np.uint8).reshape(32,32)\n        seq = np.zeros((8,3), dtype=np.float64)\n        seq[:n] = state[\'seq\']\n        if not np.isfinite(seq).all():\n            raise ValueError(\'Nonfinite sequence values\')\n        records.append({\'sample_id\': state[\'id\'], \'capture_sha256\': capture_hash,\n                        \'capture_file\': path.name, \'capture_class\': capture_class,\n                        \'expected_ustc_label\': USTC_LABELS.get(capture_class),\n                        \'label_source\': \'data/splits.py comment; capture-class mapping not independently verified\',\n                        \'image_sha256\': hashlib.sha256(image.tobytes()).hexdigest(),\n                        \'packet_indices\': state[\'packet_ids\'], \'timestamps_seconds\': state[\'times\'],\n                        \'length\': n, \'session_observed_packets\': state[\'total\'],\n                        \'capture_complete\': complete, \'policy\': POLICY, \'split\': None,\n                        \'pairing_verified\': False})\n        images.append(image)\n        sequences.append(seq)\n        lengths.append(n)\n    arrays = {\'images\': np.asarray(images, dtype=np.uint8).reshape(-1,32,32),\n              \'sequences\': np.asarray(sequences, dtype=np.float64).reshape(-1,8,3),\n              \'lengths\': np.asarray(lengths, dtype=np.int64)}\n    arrays[\'mask\'] = np.arange(8)[None,:] < arrays[\'lengths\'][:,None]\n    return records, arrays, {\'capture_file\': path.name, \'capture_sha256\': capture_hash,\n        \'class\': capture_class, \'packets_scanned\': scanned, \'excluded_packets\': excluded,\n        \'sessions\': len(records), \'complete\': complete, \'errors\': errors}\n\n\ndef annotate_matches(records, index):\n    counts = defaultdict(int)\n    for row in records:\n        counts[row[\'image_sha256\']] += 1\n    summary = defaultdict(int)\n    for row in records:\n        matches = index.get(row[\'image_sha256\'], [])\n        row[\'matching_rows\'] = matches\n        row[\'candidate_hash_multiplicity\'] = counts[row[\'image_sha256\']]\n        expected = row[\'expected_ustc_label\']\n        consistent = expected is not None and all(\n            m[\'label\'] == expected if m[\'dataset\'] == \'USTC\' else\n            m[\'label\'] == expected + 24 if m[\'dataset\'] == \'combined\' else False\n            for m in matches)\n        source_rows = [m for m in matches if m[\'dataset\'] == \'USTC\']\n        status = (\'NO_MATCH\' if not matches else \'LABEL_CONFLICT\' if not consistent else\n                  \'AMBIGUOUS_HASH\' if counts[row[\'image_sha256\']] > 1 or len(source_rows) != 1 else\n                  \'UNIQUE_IN_PILOT_ONLY\')\n        row[\'match_status\'] = status\n        summary[status] += 1\n    return dict(summary)\n\n\ndef temporal_coverage(records):\n    result = {}\n    for row in records:\n        entry = result.setdefault(row[\'capture_class\'], {\n            \'candidates\': 0, \'length_histogram\': {}, \'unique_pilot_matches\': 0,\n            \'matched_multi_packet_candidates\': 0, \'matched_with_positive_iat\': 0})\n        entry[\'candidates\'] += 1\n        length = str(row[\'length\'])\n        entry[\'length_histogram\'][length] = entry[\'length_histogram\'].get(length, 0) + 1\n        if row[\'match_status\'] == \'UNIQUE_IN_PILOT_ONLY\':\n            entry[\'unique_pilot_matches\'] += 1\n            entry[\'matched_multi_packet_candidates\'] += int(row[\'length\'] > 1)\n            times = [Decimal(t) for t in row[\'timestamps_seconds\']]\n            entry[\'matched_with_positive_iat\'] += int(any(b > a for a, b in zip(times, times[1:])))\n    return result\n\n\ndef run_pairing(npz_paths, captures, output):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    index, sources = make_index(npz_paths)\n    records, batches, scans, errors = [], [], [], []\n    for path, capture_class in captures:\n        try:\n            rows, arrays, scan = extract_candidates(path, capture_class)\n            records.extend(rows)\n            batches.append(arrays)\n            scans.append(scan)\n        except Exception as exc:\n            errors.append({\'file\': Path(path).name, \'error\': str(exc)})\n    matches = annotate_matches(records, index)\n    for i, row in enumerate(records):\n        row[\'array_row\'] = i\n    manifest = output/\'candidate_manifest.jsonl\'\n    with manifest.open(\'w\', encoding=\'utf-8\') as stream:\n        for row in records:\n            stream.write(json.dumps(row) + \'\\n\')\n    if batches:\n        np.savez_compressed(output/\'paired_candidates_NOT_TRAIN_READY.npz\',\n            **{k: np.concatenate([a[k] for a in batches]) for k in batches[0]})\n    summary = {\'schema\': \'temporal-pairing-pilot-v1\', \'policy\': POLICY,\n               \'features\': [\'transport_payload_bytes\', \'direction_first_sender_0\', \'iat_seconds\'],\n               \'normalization\': \'none; fit on future train split only\',\n               \'datasets\': sources, \'captures\': scans, \'errors\': errors,\n               \'candidates\': len(records), \'match_counts\': matches,\n               \'temporal_coverage\': temporal_coverage(records),\n               \'training_ready\': False, \'scope\': \'USTC small-capture pilot, not all scenarios\',\n               \'remaining_gates\': [\'original session/window rules and label provenance\',\n                   \'full-source collision audit (pilot uniqueness is not global uniqueness)\',\n                   \'matched sequences with multiple packets and usable timing\',\n                   \'leakage-safe split and baseline comparability\', \'Malicious_TLS raw timestamped PCAP source\'],\n               \'artifacts\': {p.name: {\'sha256\': sha_file(p), \'bytes\': p.stat().st_size}\n                             for p in output.iterdir() if p.is_file()}}\n    (output/\'pairing_summary.json\').write_text(json.dumps(summary, indent=2), encoding=\'utf-8\')\n    return summary\n', 'data/Preprocessing/utils.py': 'import numpy as np\nimport binascii\nimport scapy.all as scapy\n\n\nPACKETS_PER_FLOW = 8\nHEADER_BYTES_PER_PACKET = 80\nPAYLOAD_BYTES_PER_PACKET = 48\nBYTES_PER_PACKET = HEADER_BYTES_PER_PACKET + PAYLOAD_BYTES_PER_PACKET\nIMAGE_SIDE = 32\nIMAGE_BYTES = IMAGE_SIDE * IMAGE_SIDE\n\n\n# hex to 0-255\ndef string_to_hex_array(flow_string):\n    return np.array([int(flow_string[i:i + 2], 16) for i in range(0, len(flow_string), 2)])\n\n\ndef read_pcap_list(pcap_filename, if_augment=False, remove_ip=True, keep_payload=True):\n    """Convert eight packets into the paper\'s 32x32 grayscale input."""\n    header_hex_length = HEADER_BYTES_PER_PACKET * 2\n    payload_hex_length = PAYLOAD_BYTES_PER_PACKET * 2\n    packets = scapy.rdpcap(pcap_filename)\n    data = []\n    flow_hex_length = IMAGE_BYTES * 2\n    for packet in packets:\n        try:\n            header, payload = raw_packet_to_string(packet, remove_ip=remove_ip, keep_payload=keep_payload)\n        except ValueError:\n            # Excluded packets do not consume one of the first eight IP slots.\n            continue\n        data.append(header + payload)\n        if not if_augment and len(data) == PACKETS_PER_FLOW:\n            break\n\n    if not data:\n        return []\n\n    if not if_augment or len(data) <= PACKETS_PER_FLOW:\n        flow_string = \'\'.join(data)\n        flow_string += \'0\' * (flow_hex_length - len(flow_string))\n        flow_array = string_to_hex_array(flow_string)\n        return [{\n            "data": flow_array,\n        }]\n    else:\n        assert len(data) > PACKETS_PER_FLOW\n        flow_array_list = []\n        for i in range(len(data) - PACKETS_PER_FLOW + 1):\n            flow_string = \'\'.join(data[i:i + PACKETS_PER_FLOW])\n            flow_array_list.append(string_to_hex_array(flow_string))\n        return [{\n            "data": flow_array,\n        } for flow_array in flow_array_list]\n\n\ndef raw_packet_to_string(packet, remove_ip=True, keep_payload=True):\n    """Keep network/transport headers and application bytes as separate regions.\n\n    Read the transport payload structurally, including decoded application layers;\n    a Scapy Raw layer is not required. Never modify the caller\'s captured packet.\n    """\n    header_hex_length = HEADER_BYTES_PER_PACKET * 2\n    payload_hex_length = PAYLOAD_BYTES_PER_PACKET * 2\n    if scapy.IP in packet:\n        ip = packet[scapy.IP].copy()\n        if ip.frag or ip.flags.MF:\n            raise ValueError(\'Fragmented packets require reassembly before extraction\')\n        pad_address = \'0.0.0.0\'\n    elif scapy.IPv6 in packet:\n        ip = packet[scapy.IPv6].copy()\n        if scapy.IPv6ExtHdrFragment in ip:\n            raise ValueError(\'Fragmented packets require reassembly before extraction\')\n        pad_address = \'::\'\n    else:\n        raise ValueError(\'Non-IP packet\')\n    if scapy.TCP in ip:\n        transport = ip[scapy.TCP]\n    elif scapy.UDP in ip:\n        transport = ip[scapy.UDP]\n        if transport.sport in (67, 68, 546, 547) or transport.dport in (67, 68, 546, 547):\n            raise ValueError(\'DHCP excluded\')\n    else:\n        raise ValueError(\'Expected a TCP or UDP flow\')\n    # Remove only capture padding, not application data or transport options.\n    if scapy.Padding in ip:\n        ip[scapy.Padding].underlayer.remove_payload()\n    application_bytes = bytes(transport.payload)\n    if remove_ip:\n        ip.src, ip.dst = pad_address, pad_address\n    network_bytes = bytes(ip)\n    header_length = len(network_bytes) - len(application_bytes)\n    header = network_bytes[:header_length].hex()\n    payload = application_bytes.hex() if keep_payload else \'\'\n    header = (\n        header[:header_hex_length]\n        if len(header) > header_hex_length\n        else header + \'0\' * (header_hex_length - len(header))\n    )\n    payload = (\n        payload[:payload_hex_length]\n        if len(payload) > payload_hex_length\n        else payload + \'0\' * (payload_hex_length - len(payload))\n    )\n    return header, payload\n\n\n\nif __name__ == "__main__":\n    pass\n\n'}
SOURCE_SHA256 = {'pilot/__init__.py': 'fa09be3e956caf07db20228406db1ba733bf62f6a7dd665e1c66fb6fbcd95ef4', 'pilot/pairing.py': '5e8ad8da9ecf1e18970a85a902ff314cee683cb9e81ab1bea62cda91d53f3a3e', 'data/Preprocessing/utils.py': '4bb13d662c25a2f4086a475791640b580e090cc1253ae7ce1f08c200927752d4'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_SHA256[name]
    path = WORK / 'src' / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
sys.path.insert(0, str(WORK / 'src'))
from pilot.pairing import run_pairing, sha_file
PILOT_FOLDER = '1eRt_MeJkoVvFCMTfuHssELQ2RgqQK0Ii'
P0_REPORT_ID = '1I2Tzy_vaKYm9-8Kh1j3IMm5DC4kKppJJ'
def download(item, directory, limit):
    size = int(item.get('size', 0))
    if not 0 < size <= limit:
        raise ValueError('Missing size or download limit exceeded: ' + item['name'])
    # Drive ID as prefix prevents filename collisions and path traversal.
    path = directory / (item['id'] + '_' + Path(item['name']).name)
    path.parent.mkdir(parents=True, exist_ok=True)
    request = drive.files().get_media(fileId=item['id'], supportsAllDrives=True)
    digest = hashlib.md5()
    with path.open('wb') as handle:
        loader = MediaIoBaseDownload(handle, request, chunksize=1024**2)
        done = False
        while not done:
            _, done = loader.next_chunk(num_retries=3)
            if handle.tell() > size or handle.tell() > limit:
                raise ValueError('File grew during transfer; stop and rerun audit')
    assert path.stat().st_size == size, 'Incomplete download'
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024**2), b''):
            digest.update(block)
    if item.get('md5Checksum'):
        assert digest.hexdigest() == item['md5Checksum'], 'Drive checksum mismatch'
    return path

meta = drive.files().get(fileId=P0_REPORT_ID, fields='id,name,size,md5Checksum', supportsAllDrives=True).execute()
report_path = download(meta, WORK/'source_report', 2*1024**2)
p0 = json.loads(report_path.read_text())
assert p0['schema'] == 'temporal-pilot-audit-v1'
assert p0['missing_dataset_count'] == 0 and not p0['errors'] and not p0['download_errors']
print('P0 source run:', p0['run_id'])


In [ ]:
# Exact files discovered by P0; stale/replaced contents fail MD5 rather than silently change inputs.
expected = {'USTC_1c_train.npz', 'USTC_1c_test.npz', 'mal_32_1c_train.npz',
            'mal_32_1c_test.npz', 'combined_train_data.npz', 'combined_test_data.npz'}
datasets = [x for x in p0['dataset_inventory']['items'] if x['name'] in expected]
assert len(datasets) == 6 and {x['name'] for x in datasets} == expected
assert sum(int(x['size']) for x in datasets) <= 400*1024**2
npz_paths = []
for item in datasets:
    npz_paths.append(download(item, WORK/'npz', 400*1024**2))
    print('OK:', item['name'])
raw = p0['raw_inventory']['items']
capture_items = sorted([x for x in raw if x['relative_path'].startswith('USTC-TFC2016/')
    and x['name'].endswith('.pcap') and 0 < int(x.get('size',0)) <= 8*1024**2],
    key=lambda x: (int(x['size']), x['relative_path']))[:4]
assert capture_items, 'No small USTC PCAP in P0 report'
captures = []
for item in capture_items:
    path = download(item, WORK/'pcap', 8*1024**2)
    captures.append((path, Path(item['name']).stem))
    print('PCAP:', item['relative_path'])


In [ ]:
# Read archive directory only. Never extract/execute archive contents.
import rarfile
archive_items = [x for x in raw if x['relative_path'] == 'Malicious_TLS/malicious_TLS.rar']
archive_report = {'status': 'NOT_FOUND_IN_P0', 'entries': [], 'raw_pcap_present': None}
if len(archive_items) == 1:
    archive_path = download(archive_items[0], WORK/'archive', 20*1024**2)
    try:
        with rarfile.RarFile(archive_path) as archive:
            entries = [{'name': i.filename, 'bytes': i.file_size, 'is_dir': i.isdir()}
                       for i in archive.infolist()]
        archive_report = {'status': 'LISTED_ONLY', 'archive_sha256': sha_file(archive_path),
            'entries': entries, 'raw_pcap_present': any(
                not i['is_dir'] and i['name'].lower().endswith(('.pcap','.pcapng')) for i in entries)}
    except Exception as exc:
        archive_report = {'status': 'UNREADABLE', 'error': str(exc), 'raw_pcap_present': None}
print(json.dumps(archive_report, indent=2))
if archive_report.get('raw_pcap_present') is False:
    print('B/C: no directly listed PCAP in this archive. Need original timestamped PCAP; CSV row order is not pairing.')


In [ ]:
# No training, threshold fitting, or split assignment in this stage.
OUT = WORK/'outputs'
summary = run_pairing(npz_paths, captures, OUT)
summary.update(run_id=RUN_ID, source_sha256=SOURCE_SHA256, source_p0_sha256=sha_file(report_path),
               source_p0_run=p0['run_id'], source_pcaps=capture_items,
               malicious_tls_archive=archive_report)
(OUT/'pairing_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Candidates:', summary['candidates'])
print('Hash matching:', summary['match_counts'])
print('Temporal coverage:', json.dumps(summary['temporal_coverage'], indent=2))
if not any(x['matched_multi_packet_candidates'] for x in summary['temporal_coverage'].values()):
    print('STOP before training: no uniquely matched multi-packet sequence in this pilot. One packet has no inter-packet timing.')
print('Capture scan:', [(s['class'], s['complete'], s['sessions']) for s in summary['captures']])
print('Errors:', summary['errors'])
print('Training ready:', summary['training_ready'])
print('First-8 unmatched does NOT prove all original images unrecoverable; session/window rules still need verification.')
print('Outputs:', [(p.name, p.stat().st_size) for p in OUT.iterdir()])


## Upload hasil — cell terakhir terpisah
Membuat subfolder baru `P1_<run-id>` di folder pilot A; berisi ringkasan,
manifest kandidat, dan NPZ citra+sequence+mask. Tidak mengunggah PCAP atau checkpoint lama.
Run ulang cell ini melengkapi file yang belum ada; tidak menimpa file yang sudah ada.


In [ ]:
folder_name = 'P1_' + RUN_ID
folders = drive.files().list(q=f"'{PILOT_FOLDER}' in parents and trashed=false and name='{folder_name}'",
    fields='files(id,mimeType)', supportsAllDrives=True, includeItemsFromAllDrives=True).execute().get('files', [])
assert len(folders) <= 1, 'Ambiguous destination'
if folders:
    assert folders[0]['mimeType'] == 'application/vnd.google-apps.folder'
    destination = folders[0]['id']
else:
    destination = drive.files().create(body={'name': folder_name,
        'mimeType': 'application/vnd.google-apps.folder', 'parents': [PILOT_FOLDER]},
        fields='id', supportsAllDrives=True).execute()['id']
for path in sorted(OUT.iterdir()):
    existing = drive.files().list(q=f"'{destination}' in parents and trashed=false and name='{path.name}'",
        fields='files(id,md5Checksum)', supportsAllDrives=True, includeItemsFromAllDrives=True).execute().get('files', [])
    if existing:
        assert len(existing) == 1 and existing[0].get('md5Checksum') == hashlib.md5(path.read_bytes()).hexdigest(), 'Existing file differs; not overwritten'
        print('Already verified:', path.name)
        continue
    result = drive.files().create(body={'name': path.name, 'parents': [destination]},
        media_body=MediaFileUpload(str(path), mimetype='application/octet-stream', resumable=True),
        fields='id,md5Checksum', supportsAllDrives=True).execute()
    assert result.get('md5Checksum') == hashlib.md5(path.read_bytes()).hexdigest()
    print('Uploaded + verified:', path.name)
print('Hasil P1:', 'https://drive.google.com/drive/folders/' + destination)
